##  1 - Install

In [1]:
!pip install -q autogen-agentchat autogen-ext[openai] arxiv requests nest_asyncio sentence-transformers
print('OK')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 9.4 MB/s eta 0:00:00
OK


## 2 - Imports

In [2]:
import os, json, time, asyncio, requests, re
import arxiv, nest_asyncio
from datetime import datetime
from sentence_transformers import SentenceTransformer, util
from autogen_agentchat.agents import AssistantAgent
import autogen_agentchat.messages as acm
from autogen_ext.models.openai import OpenAIChatCompletionClient
nest_asyncio.apply()

## 3 - Groq API Key

In [5]:
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print('OK')
except Exception:
    GROQ_API_KEY = 'gsk_...'  # paste here
if not GROQ_API_KEY or GROQ_API_KEY == 'gsk_...':
    print('ERROR: Set GROQ_API_KEY at https://console.groq.com')

OK


## 4 - Model Client

In [6]:
model_client = OpenAIChatCompletionClient(
    model='llama-3.3-70b-versatile',
    api_key=GROQ_API_KEY,
    base_url='https://api.groq.com/openai/v1',
    model_capabilities={'vision':False,'function_calling':True,'json_output':False},
)
print('OK: llama-3.3-70b-versatile ready')

OK: llama-3.3-70b-versatile ready


/usr/local/lib/python3.12/dist-packages/autogen_ext/models/openai/_openai_client.py:466: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


## 5 - Data Loading, Preprocessing, Feature Extraction & Ranking

In [7]:
# STEP 1 — DATA LOADING
def fetch_arxiv_papers(queries: list, max_per_query: int = 8) -> list:
    """Load papers from arXiv (Data Loading)."""
    client = arxiv.Client()
    all_papers = []
    seen_titles = set()
    for query in queries:
        search = arxiv.Search(query=query, max_results=max_per_query,
                              sort_by=arxiv.SortCriterion.Relevance)
        count = 0
        for p in client.results(search):
            title = p.title.strip()
            if title.lower() in seen_titles:
                continue
            seen_titles.add(title.lower())
            try: year = int(p.published.year)
            except: year = 2020
            all_papers.append({
                'source': 'arXiv',
                'title': title,
                'authors': [a.name for a in p.authors[:5]],
                'year': year,
                'abstract': p.summary[:200],
                'url': p.pdf_url,
                'citation_count': 0,
            })
            count += 1
        print(f'[Data Loading] arXiv: {count} papers for: "{query}"')
    return all_papers


def fetch_semantic_scholar(query: str, max_results: int = 8) -> list:
    """Load papers from Semantic Scholar (Data Loading)."""
    url = 'https://api.semanticscholar.org/graph/v1/paper/search'
    params = {'query': query, 'limit': max_results,
              'fields': 'title,authors,year,abstract,citationCount,url'}

    SS_API_KEY = 's2k-GnNWJZGZWEHDe8hF1mIMR0wqtiIWynIQLm1wwqDZ'

    headers = {'Accept': 'application/json', 'x-api-key': SS_API_KEY}

    data = None
    for attempt in range(3):
        try:
            time.sleep(1.5)
            r = requests.get(url, params=params, headers=headers, timeout=15)
            if r.status_code == 429:
                wait = 10 * (attempt + 1)
                print(f'[SS] Rate limit (attempt {attempt+1}), waiting {wait}s...')
                time.sleep(wait)
                continue
            r.raise_for_status()
            data = r.json()
            break
        except Exception as e:
            print(f'[SS] Attempt {attempt+1} failed: {e}')

    if data is None:
        print('[SS] All attempts failed, skipping Semantic Scholar.')
        return []

    results = []
    for p in data.get('data', []):
        if not p.get('abstract'): continue
        try: year = int(str(p.get('year') or '2020')[:4])
        except: year = 2020
        results.append({
            'source': 'Semantic Scholar',
            'title': p.get('title', ''),
            'authors': [a['name'] for a in p.get('authors', [])[:5]],
            'year': year,
            'abstract': p.get('abstract', '')[:200],
            'citation_count': p.get('citationCount', 0),
            'url': p.get('url', ''),
        })
        time.sleep(1.1)
    print(f'[Data Loading] Semantic Scholar: {len(results)} papers for: "{query}"')
    return results


In [8]:
#PREPROCESSING
def preprocess_text(text: str) -> str:
    """Clean and normalize text (Preprocessing)."""
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)               # collapse whitespace
    text = re.sub(r'[^a-z0-9\s\-]', '', text)      # remove special chars
    return text


def preprocess_papers(papers: list) -> list:
    """Deduplicate and clean all papers (Preprocessing)."""
    seen, unique = set(), []
    for p in papers:
        key = preprocess_text(p.get('title', ''))
        if key and key not in seen:
            seen.add(key)
            # clean abstract and title
            p['clean_text'] = preprocess_text(
                p.get('title', '') + ' ' + p.get('abstract', '')
            )
            unique.append(p)
    print(f'[Preprocessing] {len(papers)} papers → {len(unique)} after dedup & cleaning')
    return unique

In [9]:
# STEP 3 — FEATURE EXTRACTION
print('[Feature Extraction] Loading sentence-transformer model...')
_semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
print('[Feature Extraction] Model ready: all-MiniLM-L6-v2')


def extract_features(papers: list, topic: str):
    """Generate sentence embeddings for topic and papers (Feature Extraction)."""
    topic_emb = _semantic_model.encode(topic, convert_to_tensor=True)
    for p in papers:
        text = p.get('clean_text', p.get('title', '') + ' ' + p.get('abstract', ''))
        p['embedding'] = _semantic_model.encode(text, convert_to_tensor=True)
        p['semantic_score'] = util.cos_sim(topic_emb, p['embedding']).item()
    print(f'[Feature Extraction] Embeddings computed for {len(papers)} papers')
    return papers

[Feature Extraction] Loading sentence-transformer model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[Feature Extraction] Model ready: all-MiniLM-L6-v2


In [10]:
# STEP 4 — MODELING & EVALUATION
def rank_papers(papers: list, top_n: int = 7) -> list:
    """Score and rank papers (Modeling)."""
    def score(p):
        s = p['semantic_score'] * 10                        # semantic similarity
        s += min((p.get('citation_count') or 0) / 50, 5)   # citation boost
        s += (p.get('year', 2018) - 2018) * 0.1            # recency boost
        return s
    ranked = sorted(papers, key=score, reverse=True)[:top_n]
    return ranked


def evaluate_ranking(papers: list, topic: str):
    """Print semantic scores for transparency (Evaluation)."""
    print(f'\n[Evaluation] Semantic similarity scores for topic: "{topic}"')
    print(f'{"Score":>6}  {"Year":>4}  Title')
    print('-' * 72)
    for p in papers:
        print(f"{p['semantic_score']:>6.3f}  {p.get('year','?'):>4}  {p.get('title','')[:55]}")
    avg = sum(p['semantic_score'] for p in papers) / len(papers)
    print(f'\n  Average similarity: {avg:.3f}  |  Papers: {len(papers)}')


def format_papers_for_prompt(papers: list) -> str:
    """Format paper list for LLM prompt."""
    lines = []
    for i, p in enumerate(papers, 1):
        authors = p.get('authors', [])
        if len(authors) == 1:
            author_str = authors[0].split()[-1]
        elif len(authors) == 2:
            author_str = f"{authors[0].split()[-1]} & {authors[1].split()[-1]}"
        else:
            author_str = f"{authors[0].split()[-1]} et al."
        cite_key = f"({author_str}, {p.get('year', '?')})"
        lines.append(
            f"PAPER {i}:\n"
            f"  Title: {p.get('title','')}\n"
            f"  Authors: {', '.join(authors)}\n"
            f"  Year: {p.get('year','?')}\n"
            f"  Source: {p.get('source','')}\n"
            f"  Citation key: {cite_key}\n"
            f"  Abstract: {p.get('abstract','')}\n"
            f"  URL: {p.get('url','')}\n"
        )
    return '\n'.join(lines)


def save_review(content: str, topic: str) -> str:
    """Save review to /content/"""
    slug = re.sub(r'[^a-z0-9]+', '_', topic.lower())[:40]
    filename = f'literature_review_{slug}_{datetime.now().strftime("%H%M%S")}.md'
    fp = f'/content/{filename}'
    with open(fp, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f'[Save] {fp} ({len(content.split())} words)')
    return fp


print('OK: All pipeline functions ready')

OK: All pipeline functions ready


## Step 6 - LLM Writer Agent

In [11]:
writer = AssistantAgent(
    name='Literature_Review_Writer',
    model_client=model_client,
    system_message=(
        'You are an expert academic writer specializing in literature reviews.\n'
        'You will be given a list of REAL papers with their titles, authors, years, and abstracts.\n'
        'Your job is to write a comprehensive, well-structured academic literature review.\n\n'
        'STRICT RULES:\n'
        '- ONLY cite the papers provided to you in the PAPERS LIST\n'
        '- NEVER invent or add papers not in the list\n'
        '- Use the exact citation key shown for each paper\n'
        '- Write minimum 900 words\n'
        '- Be analytical, not just descriptive\n'
        '- Compare and contrast papers within each theme\n'
    ),
)
print('Writer agent ready')

Writer agent ready


## 7- Run the Full Pipeline

In [12]:
# ============================================================
RESEARCH_TOPIC = 'Self-Attention Transformer Bert Natural Language Processing'
# More examples:
# RESEARCH_TOPIC = 'federated learning privacy preservation'
# RESEARCH_TOPIC = 'diffusion models for image generation'
# RESEARCH_TOPIC = 'graph neural networks for drug discovery'
# RESEARCH_TOPIC = 'large language models reasoning capabilities'
# ============================================================

TODAY = datetime.now().strftime('%Y-%m-%d')
print(f'Topic: {RESEARCH_TOPIC}')

# ── STEP 1: Data Loading ──────────────────────────────────────
print('\n── Step 1/4: Data Loading ──')
arxiv_papers = fetch_arxiv_papers(
    queries=[
        RESEARCH_TOPIC,
        f"{' '.join(RESEARCH_TOPIC.split()[:4])} survey review",
    ],
    max_per_query=8
)
ss_papers = fetch_semantic_scholar(RESEARCH_TOPIC, max_results=8)
raw_papers = arxiv_papers + ss_papers
print(f'[Data Loading] Total loaded: {len(raw_papers)} papers')

# ── STEP 2: Preprocessing ─────────────────────────────────────
print('\n── Step 2/4: Preprocessing ──')
clean_papers = preprocess_papers(raw_papers)

# ── STEP 3: Feature Extraction ────────────────────────────────
print('\n── Step 3/4: Feature Extraction ──')
clean_papers = extract_features(clean_papers, RESEARCH_TOPIC)

# ── STEP 4: Modeling & Evaluation ─────────────────────────────
print('\n── Step 4/4: Modeling & Evaluation ──')
top_papers = rank_papers(clean_papers, top_n=7)
evaluate_ranking(top_papers, RESEARCH_TOPIC)

# ── LLM Writing ───────────────────────────────────────────────
papers_text = format_papers_for_prompt(top_papers)
min_words = len(top_papers) * 120

prompt = f"""Write a comprehensive academic literature review on:
TOPIC: {RESEARCH_TOPIC}

Use ONLY the following {len(top_papers)} papers. Do NOT add any other references.

=== PAPERS LIST ===
{papers_text}
=== END OF PAPERS LIST ===

Write the review using this exact structure (minimum {min_words} words total):

# Literature Review: {RESEARCH_TOPIC.title()}
**Date:** {TODAY}  |  **Papers Reviewed:** {len(top_papers)}  |  **Databases:** arXiv, Semantic Scholar

## 1. Introduction
Three paragraphs providing background, significance, and scope.

## 2. Search Methodology
Describe the databases, queries used, and selection criteria.

## 3. Thematic Analysis
Group the papers into exactly 3 themes. For each theme:
- Give the theme a descriptive title
- Write 2-3 substantive paragraphs
- Cite specific papers using their citation key
- Compare and contrast the approaches

## 4. Key Findings and Trends
5-6 bullet points identifying cross-cutting trends.

## 5. Research Gaps and Future Directions
4-5 specific, actionable research gaps.

## 6. Conclusion
Two substantive paragraphs synthesizing the review.

## References
List ONLY the papers from the PAPERS LIST above in APA 7th edition format.
Do NOT add any paper not in the list.
"""

print('\nWriting review with LLM (1-2 min)...')

async def write_review():
    msgs = [acm.TextMessage(content=prompt, source='user')]
    result = None
    async for msg in writer.on_messages_stream(msgs, cancellation_token=None):
        result = msg
    return result

result = asyncio.get_event_loop().run_until_complete(write_review())

review_text = ''
if hasattr(result, 'chat_message') and result.chat_message:
    review_text = result.chat_message.content
elif hasattr(result, 'content'):
    review_text = str(result.content)

print(f'\nReview written: ~{len(review_text.split())} words')

if review_text and len(review_text) > 300:
    saved_path = save_review(review_text, RESEARCH_TOPIC)
    print(f'Saved to: {saved_path}')
else:
    print('WARNING: Review seems empty')
    print(repr(review_text[:500]))


Topic: Self-Attention Transformer Bert Natural Language Processing

── Step 1/4: Data Loading ──
[Data Loading] arXiv: 8 papers for: "Self-Attention Transformer Bert Natural Language Processing"
[Data Loading] arXiv: 7 papers for: "Self-Attention Transformer Bert Natural survey review"
[Data Loading] Semantic Scholar failed: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/search?query=Self-Attention+Transformer+Bert+Natural+Language+Processing&limit=8&fields=title%2Cauthors%2Cyear%2Cabstract%2CcitationCount%2Curl
[Data Loading] Total loaded: 15 papers

── Step 2/4: Preprocessing ──
[Preprocessing] 15 papers → 15 after dedup & cleaning

── Step 3/4: Feature Extraction ──
[Feature Extraction] Embeddings computed for 15 papers

── Step 4/4: Modeling & Evaluation ──

[Evaluation] Semantic similarity scores for topic: "Self-Attention Transformer Bert Natural Language Processing"
 Score  Year  Title
------------------------------------------------------------------

## Step 8 - Display and Download

In [13]:
from IPython.display import Markdown, display
import glob
from google.colab import files

md_files = glob.glob('/content/literature_review_*.md')
if md_files:
    latest = sorted(md_files)[-1]
    with open(latest) as f: text = f.read()
    print(f'File  : {latest}')
    print(f'Words : ~{len(text.split())}')
    print(f'Papers: {len(top_papers)}')
    print('---')
    display(Markdown(text))
    files.download(latest)
else:
    print('No .md file found')

File  : /content/literature_review_self_attention_transformer_bert_natural__214101.md
Words : ~1397
Papers: 7
---


# Literature Review: Self-Attention Transformer Bert Natural Language Processing
**Date:** 2026-05-03  |  **Papers Reviewed:** 7  |  **Databases:** arXiv, Semantic Scholar

## 1. Introduction

The field of Natural Language Processing (NLP) has witnessed significant advancements in recent years, particularly with the emergence of transformer-based models. At the heart of these models lies the self-attention mechanism, which enables the processing of input sequences in parallel, thereby enhancing computational efficiency and performance. The transformer architecture, popularized by models like BERT, has become a cornerstone in NLP tasks, demonstrating remarkable capabilities in understanding and generating human language.

The significance of self-attention in transformer models cannot be overstated. It allows the model to weigh the importance of different words in a sentence relative to each other, capturing nuanced contextual relationships that are crucial for meaningful language understanding. This capability has led to breakthroughs in various NLP tasks, including text classification, language translation, and question-answering. Moreover, the versatility of transformer models, as evidenced by their application in domains beyond text, such as vision and speech, underscores the importance of exploring and understanding the self-attention mechanism further.

The scope of this literature review encompasses a comprehensive analysis of recent research on self-attention in transformer models, with a particular focus on BERT and its applications in NLP. By examining the theoretical foundations, implementations, and applications of self-attention, this review aims to provide insights into the current state of the field, identify trends and gaps, and outline potential future directions for research. The selection of papers for this review is based on their relevance to self-attention, transformer models, and NLP, ensuring a broad coverage of theoretical, methodological, and applicational aspects.

## 2. Search Methodology

The search for relevant papers was conducted across two databases: arXiv and Semantic Scholar. The queries used included combinations of keywords such as "self-attention," "transformer models," "BERT," "natural language processing," and "deep learning." The selection criteria for papers included their publication year (with a focus on recent research), relevance to the topic of self-attention and transformer models, and the diversity of applications and methodologies presented. Seven papers were selected for this review, each contributing unique insights into the theory, application, or future directions of self-attention in NLP.

## 3. Thematic Analysis

### Theme 1: Theoretical Foundations of Self-Attention

The theoretical underpinnings of self-attention are crucial for understanding its role in transformer models. (Mehta, 2025) provides a mathematical interpretation of self-attention, connecting it to distributional semantics principles. This work offers a unified understanding of how self-attention emerges from projecting corpus-level co-occurrences, shedding light on its ability to capture contextual relationships. In contrast, (Chen et al., 2023) focus on improving self-attention by treating it as a kernel machine, proposing asymmetric kernel SVD in primal representation to enhance its efficiency and effectiveness. These papers demonstrate the ongoing effort to theoretically ground and improve self-attention mechanisms.

The theoretical exploration of self-attention also intersects with its applications in various domains. For instance, understanding how self-attention operates in vision transformers, as discussed in (Leem & Seo, 2024), can provide insights into its generalizability across different types of input data. This theme highlights the importance of continued theoretical development and analysis of self-attention to support its widespread adoption and further innovation.

### Theme 2: Applications of Self-Attention in NLP

Self-attention has been instrumental in the success of transformer models in NLP tasks. (Bao et al., 2021) introduce BEiT, a model that applies the BERT pre-training methodology to image transformers, demonstrating the potential of self-attention in vision tasks. Similarly, (Won et al., 2019) explore the use of self-attention in music tagging, showcasing its versatility beyond text processing. These applications underscore the capability of self-attention to model complex relationships in data, regardless of the domain.

The application of self-attention in low-resource languages is another critical area of research, as highlighted by (Alam et al., 2021). The utility of transformer models in Bangla NLP tasks suggests that self-attention can be effective even in languages with limited training data, provided that appropriate pre-training strategies are employed. This theme emphasizes the broad applicability of self-attention in NLP, from enriching representations in well-resourced languages to enabling NLP in low-resource settings.

### Theme 3: Future Directions and Innovations

Looking ahead, the field of self-attention and transformer models is poised for further innovation. (Chang et al., 2024) discuss the potential of prompting speech language models for speech processing tasks, illustrating how self-attention can be leveraged in multimodal contexts. The concept of prompting, which involves conditioning a model on a specific task or input, can be particularly powerful when combined with self-attention, as it allows for more focused and adaptive processing of input sequences.

The future of self-attention also depends on addressing current limitations and exploring new methodologies. For example, improving the efficiency of self-attention mechanisms, as proposed in (Chen et al., 2023), can make transformer models more viable for real-time applications and larger-scale datasets. Moreover, integrating self-attention with other attention mechanisms or architectural components can lead to more robust and flexible models, capable of handling a wider range of tasks and input types.

## 4. Key Findings and Trends

* Self-attention is a critical component of transformer models, enabling the parallel processing of input sequences and capturing nuanced contextual relationships.
* Theoretical foundations of self-attention are being continuously explored and refined, with connections to distributional semantics and kernel machines.
* Applications of self-attention extend beyond text processing to include vision, music, and speech, demonstrating its versatility and generalizability.
* Low-resource languages can benefit from self-attention and transformer models, provided appropriate pre-training and fine-tuning strategies are employed.
* Prompting and conditioning are emerging as key methodologies for leveraging self-attention in task-specific and multimodal contexts.
* Efficiency and scalability of self-attention mechanisms are areas of ongoing research, aiming to make transformer models more viable for real-time and large-scale applications.

## 5. Research Gaps and Future Directions

1. **Efficient Self-Attention Mechanisms**: Developing self-attention mechanisms that can efficiently handle long sequences and large datasets without significant computational overhead.
2. **Multimodal Applications**: Exploring the application of self-attention in multimodal contexts, such as vision-language models and speech-text models, to enable more comprehensive and interactive NLP systems.
3. **Explainability and Interpretability**: Investigating methods to make self-attention and transformer models more explainable and interpretable, which is crucial for understanding model decisions and identifying potential biases.
4. **Low-Resource Languages**: Continuing research on applying self-attention and transformer models to low-resource languages, focusing on effective pre-training, fine-tuning, and adaptation strategies.
5. **Beyond Self-Attention**: Exploring alternative or complementary attention mechanisms that can further enhance the capabilities of transformer models in NLP tasks.

## 6. Conclusion

This literature review has provided a comprehensive overview of recent research on self-attention in transformer models, particularly in the context of BERT and NLP. Through the examination of theoretical foundations, applications, and future directions, it is clear that self-attention plays a pivotal role in the success of transformer models. The versatility of self-attention, its ability to capture complex contextual relationships, and its applicability across various domains underscore its significance in the field of NLP.

As the field continues to evolve, addressing the identified research gaps and exploring new methodologies will be crucial. This includes developing more efficient self-attention mechanisms, expanding the application of self-attention to multimodal contexts, and enhancing the explainability and interpretability of transformer models. By pursuing these directions, researchers can unlock the full potential of self-attention and transformer models, leading to breakthroughs in NLP and beyond. The future of self-attention and its applications holds much promise, and continued research in this area is likely to yield significant advancements in our ability to process, understand, and generate human language.

## References

Alam, F., Hasan, A., Alam, T., Khan, A., & Tajrin, J. (2021). A Review of Bangla Natural Language Processing Tasks and the Utility of Transformer Models. arXiv. https://arxiv.org/pdf/2107.03844v3

Bao, H., Dong, L., Piao, S., & Wei, F. (2021). BEiT: BERT Pre-Training of Image Transformers. arXiv. https://arxiv.org/pdf/2106.08254v2

Chang, K.-W., Wu, H., Wang, Y.-K., Wu, Y.-K., & Shen, H. (2024). SpeechPrompt: Prompting Speech Language Models for Speech Processing Tasks. arXiv. https://arxiv.org/pdf/2408.13040v1

Chen, Y., Tao, Q., Tonin, F., & Suykens, J. A. K. (2023). Primal-Attention: Self-attention through Asymmetric Kernel SVD in Primal Representation. arXiv. https://arxiv.org/pdf/2305.19798v2

Leem, S., & Seo, H. (2024). Attention Guided CAM: Visual Explanations of Vision Transformer Guided by Self-Attention. arXiv. https://arxiv.org/pdf/2402.04563v1

Mehta, N. (2025). Self-Attention as Distributional Projection: A Unified Interpretation of Transformer Architecture. arXiv. https://arxiv.org/pdf/2511.13780v1

Won, M., Chun, S., & Serra, X. (2019). Toward Interpretable Music Tagging with Self-Attention. arXiv. https://arxiv.org/pdf/1906.04972v1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>